In [5]:

# ============================================================
# SAMSUNG HEALTHCARE IoT
# BRONZE -> SILVER DATA QUALITY PIPELINE
# AWS Glue Notebook / PySpark
# ============================================================

from pyspark.context import SparkContext
from awsglue.context import GlueContext
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
from pyspark.sql.window import Window


# ============================================================
# 1. INITIALIZE SPARK
# No JOB_NAME or custom job parameters required
# ============================================================

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session


# ============================================================
# 2. S3 PATHS
# ============================================================

BRONZE = "s3://healthcare-iot-lake/raw"
SILVER = "s3://healthcare-iot-lake/silver"
QUARANTINE = "s3://healthcare-iot-lake/quarantine"

BATCH = f"{BRONZE}/batch"
STREAMING = f"{BRONZE}/streaming"

quarantine_logs = []


# ============================================================
# 3. HELPER FUNCTIONS
# ============================================================

def missing(column_name):
    """Detect null, empty, and whitespace-only values."""
    value = F.col(column_name)

    return (
        value.isNull()
        | (F.trim(value.cast("string")) == "")
    )


def read_csv(path):
    """Read CSV files with a header."""
    return (
        spark.read
        .option("header", "true")
        .option("mode", "PERMISSIVE")
        .csv(path)
    )


def split_and_save(df, table_name, pk_col):
    """
    Separate clean and rejected records.

    Clean records:
        s3://healthcare-iot-lake/silver/<table>/

    Rejected records:
        s3://healthcare-iot-lake/quarantine/<table>/

    Audit records are returned for the unified audit log.
    """

    payload_columns = [
        c for c in df.columns
        if c != "dq_rule"
    ]

    # Hash the original record to make duplicate selection
    # deterministic when duplicate rows have different contents.
    record_hash = F.sha2(
        F.to_json(
            F.struct(*[
                F.col(c) for c in payload_columns
            ])
        ),
        256,
    )

    window = (
        Window
        .partitionBy(pk_col)
        .orderBy(record_hash.asc())
    )

    ranked = df.withColumn(
        "_row_num",
        F.row_number().over(window),
    )

    # Preserve the first validation failure.
    # Otherwise, flag repeated non-null primary keys.
    flagged = (
        ranked
        .withColumn(
            "dq_failure_reason",
            F.when(
                F.col("dq_rule").isNotNull(),
                F.col("dq_rule"),
            )
            .when(
                ~missing(pk_col)
                & (F.col("_row_num") > 1),
                F.lit("DUPLICATE_PRIMARY_KEY"),
            )
            .otherwise(F.lit(None).cast("string")),
        )
        .drop("_row_num", "dq_rule")
    )

    # Clean records
    clean_df = (
        flagged
        .filter(F.col("dq_failure_reason").isNull())
        .drop("dq_failure_reason")
    )

    # Rejected records with their original payload
    bad_df = (
        flagged
        .filter(F.col("dq_failure_reason").isNotNull())
        .withColumn(
            "source_table",
            F.lit(table_name),
        )
        .withColumn(
            "record_id",
            F.col(pk_col).cast("string"),
        )
        .withColumn(
            "raw_payload",
            F.to_json(
                F.struct(*[
                    F.col(c) for c in payload_columns
                ])
            ),
        )
    )

    # Write validated data to Silver
    clean_df.write.mode("overwrite").parquet(
        f"{SILVER}/{table_name}/"
    )

    # Write rejected records to table-specific quarantine
    bad_df.drop("raw_payload").write.mode(
        "overwrite"
    ).parquet(
        f"{QUARANTINE}/{table_name}/"
    )

    # Standardize audit schema for all datasets
    audit_df = bad_df.select(
        "source_table",
        "record_id",
        "dq_failure_reason",
        "raw_payload",
    )

    clean_count = clean_df.count()
    rejected_count = bad_df.count()

    print(
        f"{table_name}: "
        f"clean={clean_count}, "
        f"quarantined={rejected_count}"
    )

    return clean_df, audit_df


# ============================================================
# 4. WARDS
# Source: raw/batch/wards/
# ============================================================

wards_raw = read_csv(f"{BATCH}/wards/")

wards_dq = wards_raw.withColumn(
    "dq_rule",
    F.when(
        missing("ward_id"),
        "NULL_WARD_ID",
    )
    .when(
        missing("ward_name"),
        "NULL_WARD_NAME",
    )
    .otherwise(F.lit(None).cast("string")),
)

clean_wards, bad_wards = split_and_save(
    wards_dq,
    "wards",
    "ward_id",
)
quarantine_logs.append(bad_wards)


# ============================================================
# 5. BEDS
# Source: raw/batch/beds/
# ============================================================

beds_raw = read_csv(f"{BATCH}/beds/")

beds_dq = beds_raw.withColumn(
    "dq_rule",
    F.when(
        missing("bed_id"),
        "NULL_BED_ID",
    )
    .when(
        missing("ward_id"),
        "NULL_WARD_ID",
    )
    .when(
        missing("bed_status"),
        "NULL_BED_STATUS",
    )
    .when(
        F.col("bed_status").contains("MISSING_REF"),
        "BROKEN_REF_BED_STATUS",
    )
    .otherwise(F.lit(None).cast("string")),
)

clean_beds, bad_beds = split_and_save(
    beds_dq,
    "beds",
    "bed_id",
)
quarantine_logs.append(bad_beds)


# ============================================================
# 6. PATIENTS
# Source: raw/batch/patients/
# ============================================================

patients_raw = read_csv(f"{BATCH}/patients/")

patients_dq = patients_raw.withColumn(
    "dq_rule",
    F.when(
        missing("patient_id"),
        "NULL_PATIENT_ID",
    )
    .when(
        missing("name"),
        "NULL_PATIENT_NAME",
    )
    .when(
        missing("date_of_birth"),
        "NULL_DOB",
    )
    .otherwise(F.lit(None).cast("string")),
)

clean_patients, bad_patients = split_and_save(
    patients_dq,
    "patients",
    "patient_id",
)
quarantine_logs.append(bad_patients)


# ============================================================
# 7. ADMISSIONS
# Source: raw/batch/admissions/
# ============================================================

admissions_raw = read_csv(f"{BATCH}/admissions/")

admissions_dq = admissions_raw.withColumn(
    "dq_rule",
    F.when(
        missing("admission_id"),
        "NULL_ADMISSION_ID",
    )
    .when(
        missing("patient_id"),
        "NULL_PATIENT_ID",
    )
    .when(
        F.col("patient_id").contains("MISSING_REF"),
        "BROKEN_REF_PATIENT_ID",
    )
    .when(
        missing("bed_id"),
        "NULL_BED_ID",
    )
    .when(
        F.col("bed_id").contains("MISSING_REF"),
        "BROKEN_REF_BED_ID",
    )
    .when(
        missing("admitted_at"),
        "NULL_ADMITTED_AT",
    )
    .otherwise(F.lit(None).cast("string")),
)

clean_admissions, bad_admissions = split_and_save(
    admissions_dq,
    "admissions",
    "admission_id",
)
quarantine_logs.append(bad_admissions)


# ============================================================
# 8. DEVICES
# Source: raw/batch/devices/
# ============================================================

devices_raw = read_csv(f"{BATCH}/devices/")

devices_dq = devices_raw.withColumn(
    "dq_rule",
    F.when(
        missing("device_id"),
        "NULL_DEVICE_ID",
    )
    .when(
        F.col("device_id").contains("MISSING_REF"),
        "BROKEN_REF_DEVICE_ID",
    )
    .when(
        missing("device_type"),
        "NULL_DEVICE_TYPE",
    )
    .when(
        missing("model"),
        "NULL_MODEL",
    )
    .otherwise(F.lit(None).cast("string")),
)

clean_devices, bad_devices = split_and_save(
    devices_dq,
    "devices",
    "device_id",
)
quarantine_logs.append(bad_devices)


# ============================================================
# 9. DEVICE ASSIGNMENTS
# Source: raw/batch/device_assignments/
# ============================================================

assignments_raw = read_csv(
    f"{BATCH}/device_assignments/"
)

assignments_dq = assignments_raw.withColumn(
    "dq_rule",
    F.when(
        missing("assignment_id"),
        "NULL_ASSIGNMENT_ID",
    )
    .when(
        missing("device_id"),
        "NULL_DEVICE_ID",
    )
    .when(
        F.col("device_id").contains("MISSING_REF"),
        "BROKEN_REF_DEVICE_ID",
    )
    .when(
        missing("patient_id"),
        "NULL_PATIENT_ID",
    )
    .when(
        F.col("patient_id").contains("MISSING_REF"),
        "BROKEN_REF_PATIENT_ID",
    )
    .when(
        missing("admission_id"),
        "NULL_ADMISSION_ID",
    )
    .when(
        F.col("admission_id").contains("MISSING_REF"),
        "BROKEN_REF_ADMISSION_ID",
    )
    .when(
        missing("assigned_at"),
        "NULL_ASSIGNED_AT",
    )
    .otherwise(F.lit(None).cast("string")),
)

clean_assignments, bad_assignments = split_and_save(
    assignments_dq,
    "device_assignments",
    "assignment_id",
)
quarantine_logs.append(bad_assignments)



# ============================================================
# 10. DEVICE STATUS EVENTS
# Source: raw/streaming/device-status-events/
# Format: JSON
# ============================================================

events_raw = spark.read.json(
    f"{STREAMING}/device-status-events/"
)

events_dq = events_raw.withColumn(
    "dq_rule",
    F.when(
        missing("event_id"),
        F.lit("NULL_EVENT_ID"),
    )
    .when(
        missing("device_id"),
        F.lit("NULL_DEVICE_ID"),
    )
    .when(
        F.col("device_id").contains("MISSING_REF"),
        F.lit("BROKEN_REF_DEVICE_ID"),
    )
    .when(
        missing("event_type"),
        F.lit("NULL_EVENT_TYPE"),
    )
    .when(
        missing("event_time"),
        F.lit("NULL_EVENT_TIME"),
    )
    .otherwise(F.lit(None).cast("string")),
)

clean_events, bad_events = split_and_save(
    events_dq,
    "device_status_events",
    "event_id",
)
quarantine_logs.append(bad_events)




# ============================================================
# 11. VITAL READINGS
# Source: raw/streaming/vital-readings/
# Format: JSON
# ============================================================

vitals_raw = spark.read.json(
    f"{STREAMING}/vital-readings/"
)

# Normalize the metric column name
if "metric_type" in vitals_raw.columns and "metric_name" not in vitals_raw.columns:
    vitals_raw = vitals_raw.withColumnRenamed(
        "metric_type", "metric_name"
    )

# Drop only admission_id (keep partition)
vitals_raw = vitals_raw.drop("admission_id")

# Create the numeric column BEFORE using it in validation
vitals_raw = vitals_raw.withColumn(
    "_metric_num",
    F.expr("try_cast(metric_value AS DOUBLE)")
)

metric_num = F.col("_metric_num")
metric_type = F.lower(
    F.trim(F.col("metric_name").cast("string"))
)

vitals_dq = vitals_raw.withColumn(
    "dq_rule",
    F.when(
        missing("reading_id"),
        F.lit("NULL_READING_ID"),
    )
    .when(
        missing("device_id"),
        F.lit("NULL_DEVICE_ID"),
    )
    .when(
        F.col("device_id").contains("MISSING_REF"),
        F.lit("BROKEN_REF_DEVICE_ID"),
    )
    .when(
        missing("patient_id"),
        F.lit("NULL_PATIENT_ID"),
    )
    .when(
        F.col("patient_id").contains("MISSING_REF"),
        F.lit("BROKEN_REF_PATIENT_ID"),
    )
    .when(
        missing("measured_at"),
        F.lit("NULL_MEASURED_AT"),
    )
    .when(
        missing("metric_name"),
        F.lit("NULL_METRIC_TYPE"),
    )
    .when(
        missing("unit"),
        F.lit("NULL_UNIT"),
    )
    .when(
        metric_type == "unknown_metric",
        F.lit("INVALID_METRIC_TYPE"),
    )
    .when(
        F.col("unit") == "unknown_unit",
        F.lit("INVALID_UNIT"),
    )
    .when(
        ~missing("metric_value") & metric_num.isNull(),
        F.lit("INVALID_METRIC_VALUE_FORMAT"),
    )
    .when(
        metric_num < 0,
        F.lit("NEGATIVE_METRIC_VALUE"),
    )
    .when(
        metric_type.isin("spo2", "oxygen_saturation")
        & (metric_num > 100),
        F.lit("IMPOSSIBLE_SPO2_OVER_100"),
    )
    .when(
        (metric_type == "temperature")
        & ((metric_num < 25) | (metric_num > 45)),
        F.lit("IMPOSSIBLE_TEMPERATURE"),
    )
    .when(
        (metric_type == "heart_rate") & (metric_num > 220),
        F.lit("IMPOSSIBLE_HEART_RATE"),
    )
    .when(
        metric_type.isin("systolic_bp", "blood_pressure_systolic")
        & (metric_num > 250),
        F.lit("IMPOSSIBLE_SYSTOLIC_BP"),
    )
    .when(
        metric_type.isin("diastolic_bp", "blood_pressure_diastolic")
        & (metric_num > 200),
        F.lit("IMPOSSIBLE_DIASTOLIC_BP"),
    )
    .otherwise(F.lit(None).cast("string")),
).withColumn(
    "reading_status",
    F.when(
        missing("metric_value"),
        F.lit("MISSING_MEASUREMENT"),
    )
    .when(
        metric_num.isNull(),
        F.lit("INVALID_MEASUREMENT_FORMAT"),
    )
    .otherwise(F.lit("MEASUREMENT_RECEIVED")),
).drop("_metric_num")

clean_vitals, bad_vitals = split_and_save(
    vitals_dq,
    "vital_readings",
    "reading_id",
)

quarantine_logs.append(bad_vitals)

# ============================================================
# 12. UNIFIED QUARANTINE AUDIT LOG
# ============================================================

unified_quarantine_df = quarantine_logs[0]

for audit_df in quarantine_logs[1:]:
    unified_quarantine_df = (
        unified_quarantine_df.unionByName(audit_df)
    )

unified_quarantine_df.write.mode("overwrite").parquet(
    f"{QUARANTINE}/unified_removed_records/"
)


# ============================================================
# 13. FINAL SUMMARY
# ============================================================

print("================================================")
print("HEALTHCARE IoT BRONZE-TO-SILVER JOB COMPLETED")
print("================================================")
print(f"Silver output:     {SILVER}")
print(f"Quarantine output: {QUARANTINE}")
print(
    "Unified audit log: "
    f"{QUARANTINE}/unified_removed_records/"
)
print("Raw source records were not modified.")


wards: clean=58, quarantined=2
beds: clean=1536, quarantined=64
patients: clean=4810, quarantined=190
admissions: clean=290, quarantined=10
devices: clean=975, quarantined=25
device_assignments: clean=280, quarantined=20
device_status_events: clean=4928, quarantined=252
vital_readings: clean=4403, quarantined=777
HEALTHCARE IoT BRONZE-TO-SILVER JOB COMPLETED
Silver output:     s3://healthcare-iot-lake/silver
Quarantine output: s3://healthcare-iot-lake/quarantine
Unified audit log: s3://healthcare-iot-lake/quarantine/unified_removed_records/
Raw source records were not modified.


In [6]:

from pyspark.sql import functions as F

path = (
    "s3://healthcare-iot-lake/quarantine/"
    "vital_readings/"
)

bad_vitals = spark.read.parquet(path)

print("Failure reason counts:")
bad_vitals.groupBy("dq_failure_reason").count() \
    .orderBy(F.desc("count")) \
    .show(30, truncate=False)

print("Sample quarantined records:")
bad_vitals.select(
    "reading_id",
    "device_id",
    "patient_id",
    "metric_name",
    "metric_value",
    "unit",
    "measured_at",
    "dq_failure_reason"
).show(15, truncate=False)


Failure reason counts:
+---------------------------+-----+
|dq_failure_reason          |count|
+---------------------------+-----+
|IMPOSSIBLE_TEMPERATURE     |126  |
|NEGATIVE_METRIC_VALUE      |105  |
|IMPOSSIBLE_SPO2_OVER_100   |94   |
|IMPOSSIBLE_SYSTOLIC_BP     |59   |
|NULL_MEASURED_AT           |55   |
|NULL_UNIT                  |53   |
|NULL_PATIENT_ID            |51   |
|INVALID_METRIC_TYPE        |43   |
|DUPLICATE_PRIMARY_KEY      |41   |
|INVALID_METRIC_VALUE_FORMAT|40   |
|INVALID_UNIT               |37   |
|IMPOSSIBLE_HEART_RATE      |30   |
|BROKEN_REF_PATIENT_ID      |23   |
|BROKEN_REF_DEVICE_ID       |20   |
+---------------------------+-----+

Sample quarantined records:
+-------------------+---------+------------------+-----------+------------+-------+--------------------+---------------------------+
|reading_id         |device_id|patient_id        |metric_name|metric_value|unit   |measured_at         |dq_failure_reason          |
+-------------------+---------+---

In [7]:

# ============================================================
# SAMSUNG HEALTHCARE IoT
# SILVER LAYER INSPECTOR
# Print every table, schema, and 10 records
# AWS Glue Notebook / PySpark
# ============================================================

from pyspark.sql import functions as F

SILVER = "s3://healthcare-iot-lake/silver/"

# List dataset folders in the Silver layer
jvm = spark._jvm
hadoop_conf = spark._jsc.hadoopConfiguration()
silver_path = jvm.org.apache.hadoop.fs.Path(SILVER)
fs = silver_path.getFileSystem(hadoop_conf)

if not fs.exists(silver_path):
    print(f"Silver path does not exist: {SILVER}")
else:
    folders = fs.listStatus(silver_path)

    table_names = sorted([
        status.getPath().getName()
        for status in folders
        if status.isDirectory()
        and not status.getPath().getName().startswith("_")
        and not status.getPath().getName().startswith(".")
    ])

    print("=" * 80)
    print("SAMSUNG HEALTHCARE IoT - SILVER LAYER")
    print(f"Location: {SILVER}")
    print(f"Number of tables: {len(table_names)}")
    print("=" * 80)

    for table_name in table_names:
        table_path = f"{SILVER}{table_name}/"

        print("\n" + "=" * 80)
        print(f"TABLE: {table_name}")
        print(f"PATH:  {table_path}")
        print("=" * 80)

        try:
            df = spark.read.parquet(table_path)

            # Show the schema
            print("\nSCHEMA:")
            df.printSchema()

            # Show the number of records
            print(f"TOTAL RECORDS: {df.count()}")

            # Display up to 10 records
            print("\nFIRST 10 RECORDS:")
            df.show(10, truncate=False)

        except Exception as e:
            print(f"ERROR reading table '{table_name}': {e}")

    print("\n" + "=" * 80)
    print("SILVER LAYER INSPECTION COMPLETED")
    print("=" * 80)


SAMSUNG HEALTHCARE IoT - SILVER LAYER
Location: s3://healthcare-iot-lake/silver/
Number of tables: 8

TABLE: admissions
PATH:  s3://healthcare-iot-lake/silver/admissions/

SCHEMA:
root
 |-- admission_id: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- bed_id: string (nullable = true)
 |-- admitted_at: string (nullable = true)
 |-- discharged_at: string (nullable = true)
 |-- status: string (nullable = true)

TOTAL RECORDS: 290

FIRST 10 RECORDS:
+------------+----------+------+--------------------+--------------------+----------+
|admission_id|patient_id|bed_id|admitted_at         |discharged_at       |status    |
+------------+----------+------+--------------------+--------------------+----------+
|A00001      |P00001    |B0001 |2026-10-08T06:22:05Z|null                |active    |
|A00002      |P00002    |B0002 |2026-09-28T06:22:05Z|null                |active    |
|A00003      |P00003    |B0003 |2026-10-06T06:22:05Z|null                |active    |
|A00004   